# Predictive Maintenance — AI4I 2020 Dataset
### Predict Equipment Failure Risks to Reduce Downtime and Maintenance Costs in Production Environments

**Dataset:** AI4I 2020 Predictive Maintenance Dataset (UCI Machine Learning Repository) — synthetic

This notebook implements the full project workflow:
- **Steps 1–8:** objective, dataset understanding, EDA, cleaning, trend analysis, feature engineering, split, model training/comparison
- **Steps 9–13:** cross-validated stability check, threshold selection, explainability (SHAP/permutation), maintenance recommendations, final save

Shared logic (data loading, feature engineering, the preprocessing pipeline, explainability, maintenance recommendations) lives in `../common/`, and is imported here rather than duplicated, so the notebook, the `notebooks/*.py` scripts, and `../app.py` can never silently drift apart.

## Step 1 — Project Objective

**Business problem.** Unplanned equipment failure in a production environment causes downtime, emergency repair costs, scrapped work-in-progress, and safety risk.

**ML problem.** Binary classification: given current sensor readings, predict whether the machine will experience a failure.

**Prediction target.** `Machine failure` (1 = failure, 0 = no failure). The five sub-flags `TWF`, `HDF`, `PWF`, `OSF`, `RNF` describe specific failure *mechanisms* and are excluded from the model inputs — they are near-direct restatements of the target (see Step 2).

**How this reduces downtime/cost.** A live failure-risk score lets a maintenance team prioritize inspection of genuinely high-risk machines ahead of an unplanned stoppage, rather than servicing on a blind schedule.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # so `import common...` works from notebooks/

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, json

sns.set_style("whitegrid")

from common.data_prep import (
    RANDOM_STATE, load_raw_data, build_model_frame, load_and_split, make_preprocessor,
    ID_COLS, CATEGORICAL_COLS, RAW_NUMERICAL_COLS, ENGINEERED_NUMERICAL_COLS,
    FEATURE_ORDER, TARGET_COL, FAILURE_MODE_COLS, engineer_features, engineer_single_input,
)

np.random.seed(RANDOM_STATE)
df = load_raw_data()
print("Shape:", df.shape)
df.head()

## Step 2 — Dataset Understanding

In [ ]:
print("Columns:", list(df.columns))
print()
print("Dtypes:\n", df.dtypes)
print()
print("Missing values:\n", df.isnull().sum())
print()
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
df.describe()

**Column roles (verified from the data):** `UDI`/`Product ID` are identifiers (no predictive value); `Type` is categorical (L/M/H); `Air temperature [K]`, `Process temperature [K]`, `Rotational speed [rpm]`, `Torque [Nm]`, `Tool wear [min]` are numerical sensor features; `Machine failure` is the target; `TWF/HDF/PWF/OSF/RNF` are failure-mode sub-flags.

**Leakage check:** do the sub-flags almost perfectly determine the target?

In [ ]:
flag_sum = df[FAILURE_MODE_COLS].sum(axis=1)
mismatch_a = ((flag_sum > 0) & (df[TARGET_COL] == 0)).sum()
mismatch_b = ((flag_sum == 0) & (df[TARGET_COL] == 1)).sum()
print(f"Flag fired but Machine failure=0: {mismatch_a}")
print(f"No flag fired but Machine failure=1: {mismatch_b}")
print()
print("Target balance:\n", df[TARGET_COL].value_counts())
print("Failure rate: {:.2f}%".format(df[TARGET_COL].mean()*100))

Only a handful of rows disagree out of 10,000 — the sub-flags are dropped from the feature set (handled centrally in `common/data_prep.py::build_model_frame`).

## Step 3 — Historical Failure Analysis (EDA)

In [ ]:
fig, ax = plt.subplots(figsize=(5,4))
sns.countplot(x=TARGET_COL, data=df, ax=ax, palette=["#4C72B0", "#C44E52"])
ax.set_xticklabels(["No Failure (0)", "Failure (1)"])
ax.set_title("Failure vs No-Failure Distribution")
plt.tight_layout(); plt.show()

In [ ]:
fail_by_type = df.groupby("Type")[TARGET_COL].mean().sort_values(ascending=False) * 100
fig, ax = plt.subplots(figsize=(5,4))
fail_by_type.plot(kind="bar", ax=ax, color="#DD8452")
ax.set_title("Failure Rate (%) by Product Type"); ax.set_ylabel("Failure Rate (%)")
plt.tight_layout(); plt.show()
fail_by_type

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10,4))
sns.boxplot(x=TARGET_COL, y="Air temperature [K]", data=df, ax=axes[0]); axes[0].set_title("Air Temperature vs Failure")
sns.boxplot(x=TARGET_COL, y="Process temperature [K]", data=df, ax=axes[1]); axes[1].set_title("Process Temperature vs Failure")
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(5,4))
sns.boxplot(x=TARGET_COL, y="Torque [Nm]", data=df, ax=ax); ax.set_title("Torque vs Failure")
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(5,4))
sns.boxplot(x=TARGET_COL, y="Rotational speed [rpm]", data=df, ax=ax); ax.set_title("Rotational Speed vs Failure")
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(5,4))
sns.boxplot(x=TARGET_COL, y="Tool wear [min]", data=df, ax=ax); ax.set_title("Tool Wear vs Failure")
plt.tight_layout(); plt.show()

In [ ]:
corr = df[RAW_NUMERICAL_COLS + [TARGET_COL]].corr()
fig, ax = plt.subplots(figsize=(7,6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation Heatmap"); plt.tight_layout(); plt.show()

## Step 4 — Data Cleaning

In [ ]:
print("Missing values total:", df.isnull().sum().sum())
for c in RAW_NUMERICAL_COLS:
    print(f"{c}: min={df[c].min()}, max={df[c].max()}")

All ranges are physically plausible — no invalid-value correction needed.

**Outlier policy:** sensor outliers are **retained**. An unusually high torque or tool-wear reading is frequently the genuine high-risk condition this model needs to learn, not noise.

**Dropped before modeling:** `UDI`, `Product ID` (identifiers) and `TWF/HDF/PWF/OSF/RNF` (leakage).

In [ ]:
X_all, y_all = build_model_frame(df)
X_all.head()

## Step 5 — Failure Trend Analysis

In [ ]:
group_means = df.groupby(TARGET_COL)[RAW_NUMERICAL_COLS].mean()
group_means

**Data-supported findings:** failed machines show higher average torque and tool wear than healthy machines; process temperature is only slightly higher on average for failed machines; rotational speed alone is a weak standalone indicator; failure rate varies mildly by `Type`.

## Step 6 — Feature Engineering

(see `common/data_prep.py::engineer_features` — the single source of truth, also used at inference time by `app.py`)

In [ ]:
X_all[["Temp_diff_K", "Mechanical_power_W", "ToolWear_x_Torque"]].describe()

1. **`Temp_diff_K` = Process temp − Air temp** — thermal stress indicator. No leakage (raw sensors only).
2. **`Mechanical_power_W` = Torque × angular speed (rad/s)** — approximates real mechanical power. No leakage.
3. **`ToolWear_x_Torque` = Tool wear × Torque** — approximates cumulative overstrain. No leakage.

None use the target, identifiers, or failure-mode flags.

## Step 7 — Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = load_and_split()
print("Train:", X_train.shape, " Test:", X_test.shape)
print(f"Train failure rate: {y_train.mean()*100:.2f}%  Test failure rate: {y_test.mean()*100:.2f}%")

80/20 split, `random_state=42`, **stratified** (imbalanced target), X/y separated, all preprocessing fit only on the training fold inside a `Pipeline`.

## Step 8 — Classification Models

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
    # Native scikit-learn gradient boosting used instead of an external XGBoost dependency
    # for portability; performance is comparable on this dataset size.
    "Gradient Boosting (HistGB)": HistGradientBoostingClassifier(random_state=RANDOM_STATE, class_weight="balanced"),
}

results, fitted_pipelines = {}, {}
for name, clf in models.items():
    pipe = Pipeline(steps=[("preprocessor", make_preprocessor()), ("classifier", clf)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, y_proba),
    }
    fitted_pipelines[name] = pipe
    print(f"--- {name} ---")
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred, target_names=["No Failure","Failure"], zero_division=0))

results_df = pd.DataFrame(results).T.sort_values("Recall", ascending=False)
results_df

**Why Recall matters most:** a false negative means a failure happens unexpectedly — the costly downtime event this project exists to prevent. A false positive only costs an extra inspection. Model selection below is weighted toward recall, not accuracy.

In [ ]:
results_df["score"] = 0.5*results_df["Recall"] + 0.3*results_df["F1"] + 0.2*results_df["ROC_AUC"]
best_model_name = results_df["score"].idxmax()
best_pipeline = fitted_pipelines[best_model_name]
print("Selected model:", best_model_name)
results_df

In [ ]:
os.makedirs("../models", exist_ok=True)
os.makedirs("../figures", exist_ok=True)
joblib.dump(best_pipeline, "../models/predictive_maintenance_model.pkl")
print("Baseline pipeline saved (Steps 9-13 below select + save the FINAL threshold/metadata).")

---
## Steps 9–13 continue below, building on the pipeline just trained.
This mirrors `notebooks/validate_explain_finalize.py`, which performs the same steps as a
standalone, reproducible script.

## Step 9 — Validate Model Stability

Stratified 5-fold cross-validation **on the training data only**. Each fold re-fits the
full `Pipeline` (scaler + one-hot-encoder + model) from scratch on that fold's training
portion, so preprocessing statistics never see the held-out fold — no leakage across
folds, and the test set is never touched here.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"recall": "recall", "precision": "precision", "f1": "f1", "roc_auc": "roc_auc"}

cv_summary = {}
for name, clf in models.items():
    pipe = Pipeline(steps=[("preprocessor", make_preprocessor()), ("classifier", clf)])
    cv_results = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    summary = {}
    print(f"--- {name} ---")
    for metric in scoring:
        scores = cv_results[f"test_{metric}"]
        summary[metric] = {"mean": float(scores.mean()), "std": float(scores.std())}
        print(f"{metric:10s}: mean={scores.mean():.4f}  std={scores.std():.4f}")
    cv_summary[name] = summary
    print()

with open("../figures/cv_stability_results.json", "w") as f:
    json.dump(cv_summary, f, indent=2)

**Overfitting check:** compare each model's CV-mean Recall/F1/ROC-AUC above to the
single train/test-split numbers from Step 8. A CV mean close to the single-split score,
with a modest standard deviation across folds, indicates a stable result rather than a
lucky/unlucky split — re-run this cell to see the actual numbers produced on your machine.

## Step 10 — Evaluate Recall & Prediction Errors, Select a Threshold

The threshold is chosen using **cross-validated out-of-fold probabilities on the training
set only** — the test set is reserved for a single, final, honest read-out afterward.

In [ ]:
from sklearn.metrics import precision_recall_curve

best_clf = models[best_model_name]
best_pipe_template = Pipeline(steps=[("preprocessor", make_preprocessor()), ("classifier", best_clf)])
train_oof_proba = cross_val_predict(best_pipe_template, X_train, y_train, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]

candidate_thresholds = np.round(np.arange(0.05, 0.96, 0.05), 2)
rows = []
for t in candidate_thresholds:
    preds = (train_oof_proba >= t).astype(int)
    rows.append({"threshold": t,
                 "precision": precision_score(y_train, preds, zero_division=0),
                 "recall": recall_score(y_train, preds, zero_division=0),
                 "f1": f1_score(y_train, preds, zero_division=0)})
threshold_df = pd.DataFrame(rows)
threshold_df

In [ ]:
MIN_RECALL_TARGET = 0.80
eligible = threshold_df[threshold_df["recall"] >= MIN_RECALL_TARGET]
if len(eligible) > 0:
    selected_threshold = float(eligible.sort_values(["recall","f1"], ascending=[True, False]).iloc[0]["threshold"])
else:
    selected_threshold = float(threshold_df.sort_values("recall", ascending=False).iloc[0]["threshold"])

print(f"Selected operating threshold = {selected_threshold} "
      f"(lowest threshold reaching >= {MIN_RECALL_TARGET:.0%} recall on train OOF predictions, tie-broken by F1).")
print("Chosen WITHOUT looking at the test set.")

In [ ]:
test_proba = best_pipeline.predict_proba(X_test)[:, 1]
test_pred_selected = (test_proba >= selected_threshold).astype(int)

cm = confusion_matrix(y_test, test_pred_selected)
print(f"TEST SET performance @ selected threshold {selected_threshold}:")
print(cm)
print(classification_report(y_test, test_pred_selected, target_names=["No Failure","Failure"], zero_division=0))

tn, fp, fn, tp = cm.ravel()
print(f"False negatives (missed failures): {fn} of {fn+tp}")
print(f"False positives (false alarms): {fp} of {tn+fp}")

**Why recall matters:** a false negative means an unplanned stoppage on the floor — exactly what this project exists to prevent. A false positive only costs one avoidable inspection.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16,4))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["No Failure","Failure"], yticklabels=["No Failure","Failure"])
axes[0].set_title(f"Confusion Matrix @ {selected_threshold}"); axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")

fpr, tpr, _ = roc_curve(y_test, test_proba)
axes[1].plot(fpr, tpr, label=f"AUC={roc_auc_score(y_test, test_proba):.3f}")
axes[1].plot([0,1],[0,1],"k--"); axes[1].set_title("ROC Curve (test)"); axes[1].legend()

prec, rec, pr_thresh = precision_recall_curve(y_test, test_proba)
axes[2].plot(rec, prec)
idx = np.argmin(np.abs(pr_thresh - selected_threshold)) if len(pr_thresh) else None
if idx is not None:
    axes[2].scatter([rec[idx]], [prec[idx]], color="red", zorder=5, label=f"threshold={selected_threshold}")
axes[2].set_xlabel("Recall"); axes[2].set_ylabel("Precision"); axes[2].set_title("Precision-Recall (test)"); axes[2].legend()

plt.tight_layout()
plt.savefig("../figures/11_12_13_threshold_eval.png", dpi=120)
plt.show()

## Step 11 — Explainability (SHAP / permutation fallback)

**Honesty note:** this notebook was run in a sandboxed environment with no outbound
internet access, so `shap`/`lime` could not be installed here, and the SHAP code path in
`common/explainability.py` is **untested in this environment**. The permutation-based
fallback below IS tested and runs. See `common/explainability.py` docstring for details
and for the ready-to-use SHAP code path (expected to work once `pip install shap` succeeds
in a normal internet-connected environment).

In [ ]:
from common.explainability import (
    global_importance_permutation, global_importance_shap, SHAP_AVAILABLE,
    explain_local, reference_values_from,
)

print("SHAP installed here:", SHAP_AVAILABLE)

if SHAP_AVAILABLE:
    try:
        global_imp = global_importance_shap(best_pipeline, X_train, X_test)
        global_method = "shap"
    except Exception as e:
        print("SHAP failed:", e)
        global_imp = global_importance_permutation(best_pipeline, X_test, y_test)
        global_method = "permutation"
else:
    global_imp = global_importance_permutation(best_pipeline, X_test, y_test)
    global_method = "permutation"

print(f"\nGlobal importance ({global_method}):")
global_imp

In [ ]:
fig, ax = plt.subplots(figsize=(7,5))
global_imp.plot(kind="barh", ax=ax); ax.invert_yaxis()
ax.set_title(f"Global Feature Importance ({global_method})")
plt.tight_layout()
plt.savefig("../figures/14_global_importance.png", dpi=120)
plt.show()

**Interpretation note:** these values describe *association* the model has learned, not
proven physical causation — the dataset is observational, not a controlled experiment.

In [ ]:
# Local explanation for one high-risk and one low-risk test-set row
example_idx = (
    list(pd.Series(test_proba, index=X_test.index).sort_values(ascending=False).head(1).index) +
    list(pd.Series(test_proba, index=X_test.index).sort_values(ascending=True).head(1).index)
)

fig, axes = plt.subplots(1, len(example_idx), figsize=(6*len(example_idx), 4))
for ax, idx in zip(axes, example_idx):
    raw_row = X_test.loc[idx, RAW_NUMERICAL_COLS + CATEGORICAL_COLS].to_dict()
    proba, contributions, method = explain_local(best_pipeline, raw_row, X_train)
    ordered = contributions.sort_values()
    colors = ["#C44E52" if v > 0 else "#4C72B0" for v in ordered.values]
    ax.barh(ordered.index, ordered.values, color=colors)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(f"Row {idx} | P(failure)={proba:.2f} | {method}")
    actual = "Failure" if y_test.loc[idx] == 1 else "No Failure"
    print(f"Row {idx}: actual={actual}, predicted proba={proba:.3f}, method={method}")
    print(contributions, "\n")

plt.tight_layout()
plt.savefig("../figures/15_local_explanation_examples.png", dpi=120)
plt.show()

## Step 12 — Maintenance Scheduling Insights

Illustrative project recommendations only — **not** a validated industrial maintenance
standard, and not based on real maintenance-cost data.

In [ ]:
from common.maintenance import recommendation_for

LOW_MAX = 0.30
HIGH_MIN = max(0.60, selected_threshold)
print(f"Risk bands: LOW < {LOW_MAX:.0%} | MEDIUM {LOW_MAX:.0%}-{HIGH_MIN:.0%} | HIGH >= {HIGH_MIN:.0%}\n")

for p in [0.05, 0.45, 0.85]:
    cat, rec = recommendation_for(p, low_max=LOW_MAX, high_min=HIGH_MIN)
    print(f"P(failure)={p:.0%} -> {cat}: {rec}")

## Step 13 — Save the Final Pipeline, Threshold, and Metadata

In [ ]:
reference_values = reference_values_from(X_train)

metadata = {
    "best_model_name": best_model_name,
    "numerical_cols_raw": RAW_NUMERICAL_COLS,
    "engineered_numerical_cols": ENGINEERED_NUMERICAL_COLS,
    "categorical_cols": CATEGORICAL_COLS,
    "feature_order": FEATURE_ORDER,
    "target_col": TARGET_COL,
    "random_state": RANDOM_STATE,
    "cv_stability": cv_summary,
    "selected_threshold": selected_threshold,
    "risk_bands": {"low_max": LOW_MAX, "high_min": HIGH_MIN},
    "test_metrics_selected_threshold": {
        "precision": float(precision_score(y_test, test_pred_selected, zero_division=0)),
        "recall": float(recall_score(y_test, test_pred_selected, zero_division=0)),
        "f1": float(f1_score(y_test, test_pred_selected, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_test, test_proba)),
    },
    "global_explainability_method": global_method,
    "shap_available_at_build_time": SHAP_AVAILABLE,
    "reference_values": reference_values,
}

joblib.dump(best_pipeline, "../models/predictive_maintenance_model.pkl")
with open("../models/model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

# Verify the saved artifact reloads and predicts identically
reloaded = joblib.load("../models/predictive_maintenance_model.pkl")
sample = X_test.iloc[[0]]
assert abs(best_pipeline.predict_proba(sample)[0,1] - reloaded.predict_proba(sample)[0,1]) < 1e-9
print("Saved + verified: reloaded pipeline reproduces identical predictions.")
print("Final model:", best_model_name, "| threshold:", selected_threshold)

## Summary

Steps 1–13 are complete and verified in this notebook: the final pipeline, its validated
threshold, cross-validation results, and explainability outputs are saved to `../models/`
and `../figures/`. Step 14 (the Streamlit app, `../app.py`) loads this saved pipeline
directly and never retrains. Step 15 (`../tests/test_predictions.py`) is a standalone,
reproducible test suite over the saved pipeline. Steps 16–17 (deployment prep and the
final write-up) are documented in `../README.md`.